In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from pathlib import Path
import io

import numpy as np
import pandas as pd
import matplotlib as mpl
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.figure import Figure
from matplotlib.ticker import AutoMinorLocator

# --------------------------------------------------------
# OPTIONAL: Bruker OPUS reader
# pip install brukeropusreader
# --------------------------------------------------------
try:
    from brukeropusreader import read_file
    HAS_OPUS = True
except ImportError:
    HAS_OPUS = False

# --------------------------------------------------------
# OPTIONAL: spectrochempy for SPA
# pip install spectrochempy
# --------------------------------------------------------
try:
    import spectrochempy as scp
    HAS_SCP = True
except ImportError:
    HAS_SCP = False

# --------------------------------------------------------
# OPTIONAL: clipboard on Windows
# pip install pillow pywin32
# --------------------------------------------------------
try:
    from PIL import Image
    import win32clipboard
    HAS_CLIPBOARD = True
except ImportError:
    HAS_CLIPBOARD = False


# ========================================================
# GLOBAL VARIABLES
# ========================================================
current_df = None
current_file = None
current_filtered_df = None
full_xlim = None
full_ylim = None
last_title = "Spectrum"
resize_after_id = None
manual_peaks = []
toolbar = None

JOURNAL_PRESETS = {
    "ACS single": {"width": 3.25, "height": 2.45},
    "ACS double": {"width": 7.00, "height": 2.80},
    "Nature single": {"width": 3.54, "height": 2.50},
    "Nature double": {"width": 7.20, "height": 3.00},
    "Elsevier single": {"width": 3.35, "height": 2.50},
    "Elsevier double": {"width": 6.85, "height": 2.90},
    "Manual": {"width": 3.25, "height": 2.45},
}


# ========================================================
# REUSABLE STYLE
# ========================================================
def set_publication_rcparams():
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.linewidth": 1.0,
        "lines.linewidth": 1.8,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.minor.width": 0.8,
        "ytick.minor.width": 0.8,
        "xtick.major.size": 5,
        "ytick.major.size": 5,
        "xtick.minor.size": 3,
        "ytick.minor.size": 3,
        "savefig.dpi": 600,
        "savefig.bbox": "tight",
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": False,
        "legend.frameon": False,
        "mathtext.default": "regular",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })


def apply_publication_style(
    ax,
    xlabel="",
    ylabel="",
    title="",
    x_minor=2,
    y_minor=2,
    legend=False
):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=8, fontweight="bold")

    for side in ["left", "right", "bottom", "top"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(1.0)

    ax.tick_params(
        axis="both",
        which="major",
        direction="in",
        top=True,
        right=True,
        length=5,
        width=1.0,
        pad=6
    )
    ax.tick_params(
        axis="both",
        which="minor",
        direction="in",
        top=True,
        right=True,
        length=3,
        width=0.8
    )

    ax.xaxis.set_minor_locator(AutoMinorLocator(x_minor))
    ax.yaxis.set_minor_locator(AutoMinorLocator(y_minor))
    ax.grid(False)

    if legend:
        leg = ax.legend(frameon=False)
        if leg is not None:
            leg.set_frame_on(False)


set_publication_rcparams()


# ========================================================
# LABELS / LANGUAGES
# ========================================================
def get_axis_labels():
    lang = language_var.get()

    labels = {
        "English": {
            "x": r"Wavenumber (cm$^{-1}$)",
            "y": "Intensity",
            "title_default": "Spectrum",
        },
        "Español": {
            "x": r"Número de onda (cm$^{-1}$)",
            "y": "Intensidad",
            "title_default": "Espectro",
        },
        "Português": {
            "x": r"Número de onda (cm$^{-1}$)",
            "y": "Intensidade",
            "title_default": "Espectro",
        },
        "Italiano": {
            "x": r"Numero d'onda (cm$^{-1}$)",
            "y": "Intensità",
            "title_default": "Spettro",
        },
        "Français": {
            "x": r"Nombre d'onde (cm$^{-1}$)",
            "y": "Intensité",
            "title_default": "Spectre",
        },
    }
    return labels.get(lang, labels["English"])


# ========================================================
# HELPERS
# ========================================================
def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None


def get_custom_title():
    custom = entry_plot_title.get().strip()
    if custom != "":
        return custom
    if current_file is not None:
        return Path(current_file).stem
    return get_axis_labels()["title_default"]


def get_export_figure_size():
    mode = fig_format_var.get()

    if mode in JOURNAL_PRESETS and mode != "Manual":
        d = JOURNAL_PRESETS[mode]
        return d["width"], d["height"]

    try:
        w = float(entry_fig_width.get())
        h = float(entry_fig_height.get())
        if w <= 0 or h <= 0:
            raise ValueError
        return w, h
    except Exception:
        return 3.25, 2.45


def find_numeric_1d_array(obj):
    try:
        arr = np.asarray(obj).ravel()
        if arr.ndim == 1 and arr.size > 10 and np.issubdtype(arr.dtype, np.number):
            return arr
    except Exception:
        pass

    if isinstance(obj, dict):
        for _, v in obj.items():
            arr = find_numeric_1d_array(v)
            if arr is not None:
                return arr
    return None


def find_param_dict(opus_data):
    preferred_keys = [
        "AB Data Parameter",
        "TR Data Parameter",
        "ScSm Data Parameter",
        "Sample Data Parameter",
        "AB",
        "TR",
    ]

    for key in preferred_keys:
        if key in opus_data and isinstance(opus_data[key], dict):
            return opus_data[key]

    for _, value in opus_data.items():
        if isinstance(value, dict):
            keys = set(value.keys())
            if {"FXV", "LXV"}.issubset(keys) or {"FXV", "LXV", "NPT"}.issubset(keys):
                return value
    return None


# ========================================================
# FILE READERS
# ========================================================
def read_ascii_spectrum(file_path):
    df = pd.read_csv(file_path, sep=None, engine="python", comment="#", header=None)
    df = df.dropna(axis=1, how="all")

    if df.shape[1] < 2:
        raise ValueError("The file does not contain at least two numeric columns.")

    df = df.iloc[:, :2].copy()
    df.columns = ["wavenumber", "intensity"]

    df["wavenumber"] = pd.to_numeric(df["wavenumber"], errors="coerce")
    df["intensity"] = pd.to_numeric(df["intensity"], errors="coerce")
    df = df.dropna().reset_index(drop=True)

    if df.empty:
        raise ValueError("No valid numeric data were found.")

    return df


def read_opus_bruker(file_path):
    if not HAS_OPUS:
        raise ImportError(
            "brukeropusreader is not installed.\nInstall with:\npip install brukeropusreader"
        )

    opus_data = read_file(file_path)

    candidate_y_keys = ["AB", "TR", "SC", "ScSm", "IgSm", "PhSm", "Refl", "RfSm"]

    y = None
    for key in candidate_y_keys:
        if key in opus_data:
            arr = find_numeric_1d_array(opus_data[key])
            if arr is not None:
                y = arr
                break

    if y is None:
        for _, value in opus_data.items():
            arr = find_numeric_1d_array(value)
            if arr is not None:
                y = arr
                break

    if y is None:
        raise ValueError(
            "Could not identify a spectral intensity array in the OPUS file.\n"
            "Try exporting from OPUS as ASCII (.dpt or .txt)."
        )

    x = None
    params = find_param_dict(opus_data)

    if params is not None:
        fxv = safe_float(params.get("FXV"))
        lxv = safe_float(params.get("LXV"))
        npt = params.get("NPT")
        npt = int(npt) if safe_float(npt) is not None else len(y)

        if fxv is not None and lxv is not None and npt is not None:
            x = np.linspace(fxv, lxv, int(npt))

    if x is None:
        root_fxv = safe_float(opus_data.get("FXV")) if isinstance(opus_data, dict) else None
        root_lxv = safe_float(opus_data.get("LXV")) if isinstance(opus_data, dict) else None
        if root_fxv is not None and root_lxv is not None:
            x = np.linspace(root_fxv, root_lxv, len(y))

    if x is None:
        x = np.arange(len(y), dtype=float)

    n = min(len(x), len(y))
    x = np.asarray(x[:n], dtype=float)
    y = np.asarray(y[:n], dtype=float)

    df = pd.DataFrame({
        "wavenumber": x,
        "intensity": y
    }).dropna().reset_index(drop=True)

    if df.empty:
        raise ValueError("No valid OPUS data could be extracted.")

    return df


def read_spa_with_spectrochempy(file_path):
    if not HAS_SCP:
        raise ImportError(
            "spectrochempy is not installed.\nInstall with:\npip install spectrochempy"
        )

    ds = scp.read_spa(file_path)
    x = np.array(ds.x.data).ravel()
    y = np.array(ds.data).ravel()

    return pd.DataFrame({
        "wavenumber": x,
        "intensity": y
    }).dropna().reset_index(drop=True)


def load_spectrum(file_path):
    ext = Path(file_path).suffix.lower()

    if ext in [".0", ".1", ".2", ".3", ".4", ".5", ".6", ".7", ".8", ".9"]:
        return read_opus_bruker(file_path)

    if ext in [".dpt", ".txt", ".csv", ".dat", ".xy"]:
        return read_ascii_spectrum(file_path)

    if ext == ".spa":
        return read_spa_with_spectrochempy(file_path)

    raise ValueError(
        f"Unsupported extension: {ext}\n"
        f"Supported: .0-.9, .dpt, .txt, .csv, .dat, .xy, .spa"
    )


# ========================================================
# INFO
# ========================================================
def update_info(df, file_path):
    info = (
        f"File: {Path(file_path).name}\n"
        f"Points: {len(df)}\n"
        f"Spectral range: {df['wavenumber'].min():.3f} to {df['wavenumber'].max():.3f}\n"
        f"Minimum intensity: {df['intensity'].min():.6g}\n"
        f"Maximum intensity: {df['intensity'].max():.6g}"
    )
    info_text.config(state="normal")
    info_text.delete("1.0", tk.END)
    info_text.insert(tk.END, info)
    info_text.config(state="disabled")


# ========================================================
# RANGE FILTER
# ========================================================
def get_filtered_df_by_range(df, xmin=None, xmax=None):
    if xmin is None or xmax is None:
        return df.copy()

    low = min(xmin, xmax)
    high = max(xmin, xmax)

    filtered = df[(df["wavenumber"] >= low) & (df["wavenumber"] <= high)].copy()

    if filtered.empty:
        raise ValueError("The selected region contains no points.")

    return filtered


# ========================================================
# MANUAL PEAKS
# ========================================================
def nearest_point_from_x(df, x_click):
    x = df["wavenumber"].values
    y = df["intensity"].values
    idx = np.argmin(np.abs(x - x_click))
    return {
        "peak_number": idx + 1,
        "index": int(idx),
        "wavenumber": float(x[idx]),
        "intensity": float(y[idx]),
    }


def sort_manual_peaks():
    global manual_peaks
    manual_peaks = sorted(manual_peaks, key=lambda d: d["wavenumber"], reverse=True)
    for i, row in enumerate(manual_peaks, start=1):
        row["peak_number"] = i


def add_manual_peak(x_click):
    global manual_peaks

    if current_filtered_df is None or current_filtered_df.empty:
        return

    peak = nearest_point_from_x(current_filtered_df, x_click)

    try:
        tol = float(entry_click_tol.get())
    except Exception:
        tol = 8.0

    for p in manual_peaks:
        if abs(p["wavenumber"] - peak["wavenumber"]) <= tol:
            return

    manual_peaks.append(peak)
    sort_manual_peaks()


def remove_nearest_manual_peak(x_click):
    global manual_peaks

    if not manual_peaks:
        return

    distances = [abs(p["wavenumber"] - x_click) for p in manual_peaks]
    idx = int(np.argmin(distances))
    manual_peaks.pop(idx)
    sort_manual_peaks()


def clear_manual_peaks():
    global manual_peaks
    manual_peaks = []
    update_manual_peaks_text()
    redraw_current_plot()


def update_manual_peaks_text():
    peaks_text.config(state="normal")
    peaks_text.delete("1.0", tk.END)

    if not manual_peaks:
        peaks_text.insert(tk.END, "No manual peaks selected.")
    else:
        dfp = pd.DataFrame(manual_peaks)[["peak_number", "wavenumber", "intensity"]].copy()
        dfp["wavenumber"] = dfp["wavenumber"].map(lambda v: f"{v:.3f}")
        dfp["intensity"] = dfp["intensity"].map(lambda v: f"{v:.6g}")
        peaks_text.insert(tk.END, dfp.to_string(index=False))

    peaks_text.config(state="disabled")


# ========================================================
# PLOT HELPERS
# ========================================================
def get_current_plot_data():
    if current_df is None:
        return None, None, None, None, None

    title = get_custom_title()

    xmin_text = entry_xmin.get().strip()
    xmax_text = entry_xmax.get().strip()
    ymin_text = entry_ymin.get().strip()
    ymax_text = entry_ymax.get().strip()

    try:
        if xmin_text != "" and xmax_text != "":
            xmin = float(xmin_text)
            xmax = float(xmax_text)
            work_df = get_filtered_df_by_range(current_df, xmin, xmax)
            xlim = (xmin, xmax)
        else:
            work_df = current_df.copy()
            xlim = None
    except Exception:
        work_df = current_df.copy()
        xlim = None

    try:
        if ymin_text != "" and ymax_text != "":
            ymin = float(ymin_text)
            ymax = float(ymax_text)
            ylim = (ymin, ymax)
        else:
            ylim = None
    except Exception:
        ylim = None

    return work_df, xlim, ylim, title, None


def get_export_title():
    return get_custom_title()


def get_visible_manual_peaks(df):
    if df is None or df.empty or not manual_peaks:
        return []

    low = min(df["wavenumber"].min(), df["wavenumber"].max())
    high = max(df["wavenumber"].min(), df["wavenumber"].max())
    return [p for p in manual_peaks if low <= p["wavenumber"] <= high]


def select_peaks_for_export(peaks, max_labels=8):
    if not peaks:
        return []
    peaks_sorted = sorted(peaks, key=lambda p: p["intensity"], reverse=True)
    selected = peaks_sorted[:max_labels]
    return sorted(selected, key=lambda p: p["wavenumber"], reverse=True)


def annotate_peak_labels(ax, df, peak_positions, max_labels=8):
    if peak_positions is None or len(peak_positions) == 0:
        return

    x = df["wavenumber"].to_numpy()
    y = df["intensity"].to_numpy()

    positions = list(peak_positions)[:max_labels]

    peaks_xy = []
    for p in positions:
        idx = np.argmin(np.abs(x - p))
        peaks_xy.append((x[idx], y[idx]))

    peaks_xy.sort(key=lambda t: t[0])

    base_dy = 10
    min_vertical_gap = 14

    placed = []

    for xp, yp in peaks_xy:
        dy = base_dy

        # Regla principal: si están entre 1 y 30 cm^-1, subir la altura
        for px, py, pdy in placed:
            dx_cm = abs(xp - px)
            if 1 <= dx_cm <= 30:
                dy = max(dy, pdy + min_vertical_gap)

        # Chequeo extra para evitar que dos alturas iguales se toquen
        changed = True
        while changed:
            changed = False
            for px, py, pdy in placed:
                dx_cm = abs(xp - px)
                if 1 <= dx_cm <= 30 and abs(dy - pdy) < min_vertical_gap:
                    dy = pdy + min_vertical_gap
                    changed = True

        placed.append((xp, yp, dy))

        ax.annotate(
            f"{xp:.1f}",
            xy=(xp, yp),
            xytext=(0, dy),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7,
            color="black",
            bbox=dict(
                boxstyle="round,pad=0.08",
                fc="white",
                ec="none",
                alpha=0.95
            ),
            arrowprops=dict(
                arrowstyle="-",
                lw=0.7,
                color="0.5",
                shrinkA=0,
                shrinkB=0,
                connectionstyle="arc3,rad=0"
            ),
            zorder=5,
            clip_on=False
        )


def draw_spectrum_publication(
    ax,
    df,
    x_col="wavenumber",
    y_col="intensity",
    xlabel=r"Wavenumber (cm$^{-1}$)",
    ylabel="Intensity",
    title="",
    xlim=None,
    ylim=None,
    peak_positions=None,
    export_mode=False,
    experimental=False
):
    ax.clear()

    x = df[x_col].to_numpy()
    y = df[y_col].to_numpy()

    linestyle_main = "--" if experimental else "-"
    linewidth_main = 1.6 if export_mode else 1.2

    ax.plot(
        x,
        y,
        linestyle=linestyle_main,
        linewidth=linewidth_main,
        color="black",
        solid_capstyle="round",
        zorder=2,
        label=None
    )

    if xlim is not None:
        ax.set_xlim(xlim)

    if ylim is not None:
        ax.set_ylim(ylim)

    if len(x) > 1 and x[0] < x[-1]:
        ax.invert_xaxis()

    if peak_positions is not None and len(peak_positions) > 0:
        for p in peak_positions:
            ax.axvline(
                x=p,
                color="0.45",
                linestyle=(0, (4, 3)),
                linewidth=0.9,
                alpha=0.9,
                zorder=1
            )

    apply_publication_style(
        ax,
        xlabel=xlabel,
        ylabel=ylabel,
        title=title,
        x_minor=2,
        y_minor=2,
        legend=False
    )

    ax.figure.tight_layout()


def redraw_interface_plot(ax, canvas, df, title="", xlim=None, ylim=None, peak_positions=None):
    labels = get_axis_labels()

    draw_spectrum_publication(
        ax=ax,
        df=df,
        x_col="wavenumber",
        y_col="intensity",
        xlabel=labels["x"],
        ylabel=labels["y"],
        title=title,
        xlim=xlim,
        ylim=ylim,
        peak_positions=peak_positions,
        export_mode=False,
        experimental=False
    )

    annotate_peak_labels(ax, df, peak_positions)
    ax.figure.tight_layout()
    canvas.draw_idle()


def export_publication_figure(
    output_path,
    df,
    fig_width,
    fig_height,
    title="",
    xlim=None,
    ylim=None,
    peak_positions=None,
    xlabel=r"Wavenumber (cm$^{-1}$)",
    ylabel="Intensity",
    experimental=False
):
    export_fig = Figure(figsize=(fig_width, fig_height), dpi=300)
    export_ax = export_fig.add_subplot(111)

    draw_spectrum_publication(
        ax=export_ax,
        df=df,
        x_col="wavenumber",
        y_col="intensity",
        xlabel=xlabel,
        ylabel=ylabel,
        title=title,
        xlim=xlim,
        ylim=ylim,
        peak_positions=peak_positions,
        export_mode=True,
        experimental=experimental
    )

    annotate_peak_labels(export_ax, df, peak_positions)
    export_fig.tight_layout()

    ext = str(output_path).lower()
    if ext.endswith(".svg"):
        export_fig.savefig(output_path, format="svg", facecolor="white", transparent=False)
    elif ext.endswith(".png"):
        export_fig.savefig(output_path, format="png", dpi=600, facecolor="white", transparent=False)
    elif ext.endswith(".pdf"):
        export_fig.savefig(output_path, format="pdf", facecolor="white", transparent=False)
    else:
        export_fig.savefig(output_path, dpi=600, facecolor="white", transparent=False)


# ========================================================
# FIT / REFRESH
# ========================================================
def fit_figure_to_plot_frame():
    root.update_idletasks()

    frame_w = max(plot_frame.winfo_width() - 10, 400)
    frame_h = max(plot_frame.winfo_height() - 25, 260)

    dpi = fig.get_dpi()
    width_in = frame_w / dpi
    height_in = frame_h / dpi

    fig.set_size_inches(width_in, height_in, forward=True)
    canvas_widget.configure(width=frame_w, height=frame_h)


def plot_spectrum(df, title="", xlim=None, ylim=None):
    global full_xlim, full_ylim, current_filtered_df, last_title

    current_filtered_df = df.copy()
    last_title = title if title else get_axis_labels()["title_default"]

    fit_figure_to_plot_frame()

    peak_positions = [p["wavenumber"] for p in get_visible_manual_peaks(df)]

    redraw_interface_plot(
        ax=ax,
        canvas=canvas,
        df=df,
        title=last_title,
        xlim=xlim,
        ylim=ylim,
        peak_positions=peak_positions
    )

    x = df["wavenumber"].values
    y = df["intensity"].values
    full_xlim = (np.min(x), np.max(x))
    full_ylim = (np.min(y), np.max(y))


def redraw_current_plot():
    if current_df is None:
        ax.clear()
        labels = get_axis_labels()
        apply_publication_style(ax, xlabel=labels["x"], ylabel=labels["y"], title=labels["title_default"])
        fig.tight_layout()
        canvas.draw_idle()
        return

    work_df, xlim, ylim, title, _ = get_current_plot_data()
    plot_spectrum(work_df, title=title, xlim=xlim, ylim=ylim)


def auto_refresh_plot():
    global resize_after_id
    resize_after_id = None
    redraw_current_plot()


def schedule_auto_refresh(delay=120):
    global resize_after_id
    if resize_after_id is not None:
        try:
            root.after_cancel(resize_after_id)
        except Exception:
            pass
    resize_after_id = root.after(delay, auto_refresh_plot)


# ========================================================
# TOOLBAR CONTROL
# ========================================================
def disable_toolbar_mode():
    global toolbar
    if toolbar is None:
        return

    try:
        mode = toolbar.mode
        if mode == "zoom rect":
            toolbar.zoom()
        elif mode == "pan/zoom":
            toolbar.pan()
    except Exception:
        pass


def on_mouse_release(event):
    if toolbar is None:
        return

    try:
        if toolbar.mode in ["zoom rect", "pan/zoom"]:
            root.after(80, disable_toolbar_mode)
    except Exception:
        pass


# ========================================================
# INTERACTION
# ========================================================
def on_plot_click(event):
    if current_df is None:
        return

    if event.inaxes != ax or event.xdata is None:
        return

    if toolbar is not None and toolbar.mode != "":
        return

    if event.button == 1:
        add_manual_peak(event.xdata)
        update_manual_peaks_text()
        redraw_current_plot()
    elif event.button == 3:
        remove_nearest_manual_peak(event.xdata)
        update_manual_peaks_text()
        redraw_current_plot()


# ========================================================
# ACTIONS
# ========================================================
def open_file():
    global current_df, current_file, manual_peaks

    file_path = filedialog.askopenfilename(
        title="Select spectrum",
        filetypes=[
            ("Bruker OPUS", "*.0 *.1 *.2 *.3 *.4 *.5 *.6 *.7 *.8 *.9"),
            ("Bruker ASCII / text", "*.dpt *.txt *.csv *.dat *.xy"),
            ("SPA files", "*.spa"),
            ("All supported", "*.0 *.1 *.2 *.3 *.4 *.5 *.6 *.7 *.8 *.9 *.dpt *.txt *.csv *.dat *.xy *.spa"),
            ("All files", "*.*"),
        ]
    )

    if not file_path:
        return

    try:
        df = load_spectrum(file_path)

        current_df = df
        current_file = file_path
        manual_peaks = []

        update_info(df, file_path)

        if entry_plot_title.get().strip() == "":
            entry_plot_title.delete(0, tk.END)
            entry_plot_title.insert(0, Path(file_path).stem)

        entry_xmin.delete(0, tk.END)
        entry_xmin.insert(0, f"{df['wavenumber'].min():.2f}")

        entry_xmax.delete(0, tk.END)
        entry_xmax.insert(0, f"{df['wavenumber'].max():.2f}")

        entry_ymin.delete(0, tk.END)
        entry_ymin.insert(0, f"{df['intensity'].min():.4f}")

        entry_ymax.delete(0, tk.END)
        entry_ymax.insert(0, f"{df['intensity'].max():.4f}")

        plot_spectrum(df, title=get_custom_title())
        update_manual_peaks_text()

        schedule_auto_refresh(50)

    except Exception as e:
        messagebox.showerror("Error", str(e))


def apply_range():
    if current_df is None:
        messagebox.showwarning("Warning", "You must open a file first.")
        return

    xmin_text = entry_xmin.get().strip()
    xmax_text = entry_xmax.get().strip()
    ymin_text = entry_ymin.get().strip()
    ymax_text = entry_ymax.get().strip()

    try:
        if xmin_text != "" and xmax_text != "":
            xmin = float(xmin_text)
            xmax = float(xmax_text)
        else:
            xmin, xmax = None, None

        if ymin_text != "" and ymax_text != "":
            ymin = float(ymin_text)
            ymax = float(ymax_text)
            ylim = (ymin, ymax)
        else:
            ylim = None

    except ValueError:
        messagebox.showerror("Error", "x/y min and max must be numeric.")
        return

    try:
        if xmin is not None and xmax is not None:
            filtered_df = get_filtered_df_by_range(current_df, xmin, xmax)
            plot_spectrum(filtered_df, title=get_custom_title(), xlim=(xmin, xmax), ylim=ylim)
        else:
            plot_spectrum(current_df, title=get_custom_title(), ylim=ylim)

    except Exception as e:
        messagebox.showerror("Error", str(e))


def reset_range():
    if current_df is None:
        return

    plot_spectrum(current_df, title=get_custom_title())

    entry_xmin.delete(0, tk.END)
    entry_xmin.insert(0, f"{current_df['wavenumber'].min():.2f}")

    entry_xmax.delete(0, tk.END)
    entry_xmax.insert(0, f"{current_df['wavenumber'].max():.2f}")

    entry_ymin.delete(0, tk.END)
    entry_ymin.insert(0, f"{current_df['intensity'].min():.4f}")

    entry_ymax.delete(0, tk.END)
    entry_ymax.insert(0, f"{current_df['intensity'].max():.4f}")


def save_csv():
    if current_df is None:
        messagebox.showwarning("Warning", "You must open a file first.")
        return

    initial_name = Path(current_file).with_suffix(".csv").name
    output_path = filedialog.asksaveasfilename(
        title="Save CSV",
        defaultextension=".csv",
        initialfile=initial_name,
        filetypes=[("CSV files", "*.csv")]
    )

    if not output_path:
        return

    try:
        current_df.to_csv(output_path, index=False)
        messagebox.showinfo("Success", f"File saved to:\n{output_path}")
    except Exception as e:
        messagebox.showerror("Error", str(e))


def save_manual_peaks_csv():
    if not manual_peaks:
        messagebox.showwarning("Warning", "There are no manual peaks to export.")
        return

    initial_name = Path(current_file).stem + "_manual_peaks.csv"
    output_path = filedialog.asksaveasfilename(
        title="Save manual peak table",
        defaultextension=".csv",
        initialfile=initial_name,
        filetypes=[("CSV files", "*.csv")]
    )

    if not output_path:
        return

    try:
        pd.DataFrame(manual_peaks).to_csv(output_path, index=False)
        messagebox.showinfo("Success", f"Peak table saved to:\n{output_path}")
    except Exception as e:
        messagebox.showerror("Error", str(e))


def save_figure():
    if current_df is None:
        messagebox.showwarning("Warning", "You must open a file first.")
        return

    initial_name = Path(current_file).stem + ".png"
    output_path = filedialog.asksaveasfilename(
        title="Save figure",
        defaultextension=".png",
        initialfile=initial_name,
        filetypes=[
            ("PNG", "*.png"),
            ("SVG", "*.svg"),
            ("PDF", "*.pdf"),
            ("TIFF", "*.tif"),
            ("All files", "*.*"),
        ]
    )

    if not output_path:
        return

    try:
        work_df, xlim, ylim, _, _ = get_current_plot_data()
        if work_df is None or work_df.empty:
            raise ValueError("No data available for export.")

        fig_w, fig_h = get_export_figure_size()
        peak_positions = [p["wavenumber"] for p in select_peaks_for_export(get_visible_manual_peaks(work_df), max_labels=8)]
        labels = get_axis_labels()

        export_publication_figure(
            output_path=output_path,
            df=work_df,
            fig_width=fig_w,
            fig_height=fig_h,
            title=get_export_title(),
            xlim=xlim,
            ylim=ylim,
            peak_positions=peak_positions,
            xlabel=labels["x"],
            ylabel=labels["y"],
            experimental=False
        )

        messagebox.showinfo("Success", f"Figure saved to:\n{output_path}")

    except Exception as e:
        messagebox.showerror("Error", str(e))


def copy_figure_to_clipboard():
    if current_df is None:
        messagebox.showwarning("Warning", "You must open a file first.")
        return

    if not HAS_CLIPBOARD:
        messagebox.showerror(
            "Error",
            "Clipboard packages are not installed.\nInstall with:\npip install pillow pywin32"
        )
        return

    try:
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=600, bbox_inches="tight")
        buf.seek(0)

        image = Image.open(buf).convert("RGB")
        output = io.BytesIO()
        image.save(output, "BMP")
        data = output.getvalue()[14:]
        output.close()

        win32clipboard.OpenClipboard()
        win32clipboard.EmptyClipboard()
        win32clipboard.SetClipboardData(win32clipboard.CF_DIB, data)
        win32clipboard.CloseClipboard()

        messagebox.showinfo("Success", "Figure copied to clipboard.")
    except Exception as e:
        messagebox.showerror("Error", str(e))


def on_language_change(event=None):
    redraw_current_plot()


def on_figure_format_change(event=None):
    mode = fig_format_var.get()

    if mode == "Manual":
        entry_fig_width.config(state="normal")
        entry_fig_height.config(state="normal")
        manual_hint_var.set("Manual export size")
    else:
        entry_fig_width.config(state="disabled")
        entry_fig_height.config(state="disabled")
        d = JOURNAL_PRESETS[mode]
        manual_hint_var.set(f"Export size: {d['width']:.2f} × {d['height']:.2f} in")

    schedule_auto_refresh(50)


def on_manual_size_apply():
    schedule_auto_refresh(50)


def on_plot_title_change(event=None):
    redraw_current_plot()


def on_plot_frame_resize(event):
    schedule_auto_refresh(120)


# ========================================================
# GUI
# ========================================================
root = tk.Tk()
root.title("Interactive Bruker FTIR spectrum picker")
root.geometry("1380x960")

language_var = tk.StringVar(value="English")
fig_format_var = tk.StringVar(value="ACS single")
manual_hint_var = tk.StringVar(value="Export size: 3.25 × 2.45 in")

top_frame = tk.Frame(root)
top_frame.pack(fill="x", padx=10, pady=10)

btn_open = tk.Button(top_frame, text="Open file", command=open_file, width=14)
btn_open.pack(side="left", padx=4)

btn_save_csv = tk.Button(top_frame, text="Save CSV", command=save_csv, width=14)
btn_save_csv.pack(side="left", padx=4)

btn_save_fig = tk.Button(top_frame, text="Save figure", command=save_figure, width=14)
btn_save_fig.pack(side="left", padx=4)

btn_copy_fig = tk.Button(top_frame, text="Copy figure", command=copy_figure_to_clipboard, width=14)
btn_copy_fig.pack(side="left", padx=4)

btn_exit_nav = tk.Button(top_frame, text="Exit zoom/pan", command=disable_toolbar_mode, width=14)
btn_exit_nav.pack(side="left", padx=4)

style_frame = tk.LabelFrame(root, text="Figure and label options")
style_frame.pack(fill="x", padx=10, pady=5)

tk.Label(style_frame, text="Axis language:").pack(side="left", padx=5)

cmb_language = ttk.Combobox(
    style_frame,
    textvariable=language_var,
    values=["English", "Español", "Português", "Italiano", "Français"],
    state="readonly",
    width=12
)
cmb_language.pack(side="left", padx=5)
cmb_language.bind("<<ComboboxSelected>>", on_language_change)

tk.Label(style_frame, text="Journal format:").pack(side="left", padx=10)

cmb_format = ttk.Combobox(
    style_frame,
    textvariable=fig_format_var,
    values=[
        "ACS single", "ACS double",
        "Nature single", "Nature double",
        "Elsevier single", "Elsevier double",
        "Manual"
    ],
    state="readonly",
    width=16
)
cmb_format.pack(side="left", padx=5)
cmb_format.bind("<<ComboboxSelected>>", on_figure_format_change)

tk.Label(style_frame, text="Width (in):").pack(side="left", padx=8)
entry_fig_width = tk.Entry(style_frame, width=8)
entry_fig_width.pack(side="left", padx=3)
entry_fig_width.insert(0, "3.25")
entry_fig_width.config(state="disabled")

tk.Label(style_frame, text="Height (in):").pack(side="left", padx=8)
entry_fig_height = tk.Entry(style_frame, width=8)
entry_fig_height.pack(side="left", padx=3)
entry_fig_height.insert(0, "2.45")
entry_fig_height.config(state="disabled")

btn_refresh_style = tk.Button(style_frame, text="Update preview", command=on_manual_size_apply)
btn_refresh_style.pack(side="left", padx=10)

lbl_hint = tk.Label(style_frame, textvariable=manual_hint_var, fg="gray30")
lbl_hint.pack(side="left", padx=12)

title_frame = tk.LabelFrame(root, text="Image title")
title_frame.pack(fill="x", padx=10, pady=5)

tk.Label(title_frame, text="Title:").pack(side="left", padx=5)
entry_plot_title = tk.Entry(title_frame, width=60)
entry_plot_title.pack(side="left", padx=5, fill="x", expand=True)
entry_plot_title.bind("<KeyRelease>", on_plot_title_change)

btn_clear_title = tk.Button(
    title_frame,
    text="Use file name",
    command=lambda: [entry_plot_title.delete(0, tk.END), redraw_current_plot()]
)
btn_clear_title.pack(side="left", padx=8)

range_frame = tk.LabelFrame(root, text="Plot range")
range_frame.pack(fill="x", padx=10, pady=5)

tk.Label(range_frame, text="xmin:").pack(side="left", padx=5)
entry_xmin = tk.Entry(range_frame, width=10)
entry_xmin.pack(side="left", padx=5)

tk.Label(range_frame, text="xmax:").pack(side="left", padx=5)
entry_xmax = tk.Entry(range_frame, width=10)
entry_xmax.pack(side="left", padx=5)

tk.Label(range_frame, text="ymin:").pack(side="left", padx=12)
entry_ymin = tk.Entry(range_frame, width=10)
entry_ymin.pack(side="left", padx=5)

tk.Label(range_frame, text="ymax:").pack(side="left", padx=5)
entry_ymax = tk.Entry(range_frame, width=10)
entry_ymax.pack(side="left", padx=5)

btn_apply = tk.Button(range_frame, text="Apply range", command=apply_range)
btn_apply.pack(side="left", padx=8)

btn_reset = tk.Button(range_frame, text="Reset", command=reset_range)
btn_reset.pack(side="left", padx=8)

manual_frame = tk.LabelFrame(root, text="Manual peak selection")
manual_frame.pack(fill="x", padx=10, pady=5)

tk.Label(manual_frame, text="Duplicate tolerance (cm⁻¹):").pack(side="left", padx=5)
entry_click_tol = tk.Entry(manual_frame, width=8)
entry_click_tol.pack(side="left", padx=5)
entry_click_tol.insert(0, "8")

btn_clear_manual = tk.Button(manual_frame, text="Clear selected peaks", command=clear_manual_peaks)
btn_clear_manual.pack(side="left", padx=10)

btn_save_manual = tk.Button(manual_frame, text="Export manual peaks", command=save_manual_peaks_csv)
btn_save_manual.pack(side="left", padx=10)

info_frame = tk.LabelFrame(root, text="Spectrum information")
info_frame.pack(fill="x", padx=10, pady=5)

info_text = tk.Text(info_frame, height=6, wrap="word")
info_text.pack(fill="x", padx=5, pady=5)
info_text.config(state="disabled")

results_frame = tk.LabelFrame(root, text="Selected manual peaks")
results_frame.pack(fill="x", padx=10, pady=5)

peaks_text = tk.Text(results_frame, height=8, wrap="none")
peaks_text.pack(fill="x", padx=5, pady=5)
peaks_text.config(state="disabled")

plot_frame = tk.LabelFrame(root, text="Interactive plot")
plot_frame.pack(fill="both", expand=True, padx=10, pady=10)
plot_frame.bind("<Configure>", on_plot_frame_resize)

toolbar_frame = tk.Frame(plot_frame)
toolbar_frame.pack(fill="x", side="top")

canvas_frame = tk.Frame(plot_frame)
canvas_frame.pack(fill="both", expand=True, side="top")

fig = Figure(figsize=(3.25, 2.45), dpi=100)
ax = fig.add_subplot(111)
labels = get_axis_labels()
apply_publication_style(ax, xlabel=labels["x"], ylabel=labels["y"], title=labels["title_default"])

canvas = FigureCanvasTkAgg(fig, master=canvas_frame)
canvas_widget = canvas.get_tk_widget()
canvas_widget.pack(fill="both", expand=True)

toolbar = NavigationToolbar2Tk(canvas, toolbar_frame)
toolbar.update()

canvas.mpl_connect("button_press_event", on_plot_click)
canvas.mpl_connect("button_release_event", on_mouse_release)

def initial_refresh():
    redraw_current_plot()

root.after(200, initial_refresh)
root.mainloop()